# Configs 튜토리얼: 목록 + 기능 이해

이 노트북은 `src/garak/configs/*.yaml` 파일을 **처음 보는 사람**이 빠르게 이해하도록 만든 가이드입니다.

## 목표
- 어떤 config 파일이 있는지 한 번에 본다.
- 각 config의 핵심 설정(`system`, `run`, `plugins`)을 읽는다.
- 내 목적(빠른 점검/폭넓은 점검/toxicity 집중)에 맞는 파일을 고른다.

> 코드 원칙: 최소 코드 + 충분한 설명


## 1) 8개 config를 한눈에 보는 비교 표

아래 표는 `src/garak/configs/*.yaml` 8개 파일을 한 번에 비교합니다.

- `기능 설명`: 파일명/주석/설정값을 기반으로 한 요약
- `seed 종류`: `seed_spec`에서 어떤 seed 라인업을 쓰는지 요약
- `seed 개수`: 쉼표 기반 seed 항목 개수(참고용)
- `주요 설정`: `lite`, `generations`, `soft_seed_prompt_cap`, `attacker_spec`, `extended_judges` 등
- `세부 기능`: payload/parallel 설정 같은 실행 디테일

초보자는 먼저 `fast.yaml`, `broad.yaml`, `tox_and_attackers.yaml` 3개를 비교해서 시작하면 이해가 빠릅니다.


In [5]:
from pathlib import Path
import os
import json
import yaml
from IPython.display import Markdown, display

In [ ]:
# ------------------------------------------------------------
# 1) repo 루트 자동 탐색
# ------------------------------------------------------------
# 노트북은 보통 tutorials/ 폴더에서 열리기 때문에,
# 상대경로로 바로 garak/configs를 찾으면 실패할 수 있습니다.
# 그래서 현재 경로에서 상위로 올라가며 repo 루트를 찾습니다.
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for base in [start, *start.parents]:
        has_configs = (base / "garak" / "configs").exists()
        has_resources = (base / "garak" / "resources").exists()
        if has_configs and has_resources:
            return base
    raise FileNotFoundError(f"repo 루트를 찾지 못했습니다. cwd={Path.cwd()}")


repo_root = find_repo_root(Path.cwd())
cfg_dir = repo_root / "garak" / "configs"
cache_file = repo_root / "garak" / "resources" / "plugin_cache.json"

# configs 폴더의 yaml 파일 목록(= preset 목록)
presets = sorted(cfg_dir.glob("*.yaml"))
print("Available presets:")
for p in presets:
    print(" -", p.name)


Available presets:
 - bag.yaml
 - broad.yaml
 - default.yaml
 - fast.yaml
 - full.yaml
 - long_attack_gen.yaml
 - notox.yaml
 - tox_and_attackers.yaml


In [4]:
# ------------------------------------------------------------
# 2) plugin_cache.json에서 seed plugin 인덱스 로드
# ------------------------------------------------------------
# 목적:
# - 각 config의 seed_spec("encoding,dan,...")을 실제 seed 목록으로 확장해
#   몇 개 seed가 실행되는지 보여주기 위함
# 주의:
# - cache 파일이 없으면 확장 없이 raw token 기준으로만 표시합니다.
seed_plugins = {}
if cache_file.exists():
    pc = json.loads(cache_file.read_text(encoding="utf-8"))
    if isinstance(pc, dict) and isinstance(pc.get("seeds"), dict):
        seed_plugins = pc["seeds"]

seed_keys = sorted(seed_plugins.keys())


def parse_seed_spec(value):
    # seed_spec은 보통 문자열("a,b,c")이지만,
    # 리스트로 들어오는 경우도 있어 둘 다 처리합니다.
    if value is None:
        return []
    if isinstance(value, str):
        return [t.strip() for t in value.split(",") if t.strip()]
    if isinstance(value, list):
        out = []
        for v in value:
            if isinstance(v, str):
                out += [t.strip() for t in v.split(",") if t.strip()]
        return out
    return []


def resolve_seed_token(token: str):
    """
    token 예:
      - "encoding" (family)
      - "grandma.Win10" (specific seed)

    반환:
      - ("family", [seed_key,...])
      - ("direct", [seed_key])
      - ("raw", [])  # cache가 없어서 확장 불가
    """
    if not seed_plugins:
        return ("raw", [])

    # 점(.)이 있으면 일반적으로 단일 seed로 간주
    if "." in token:
        key = token if token.startswith("seeds.") else f"seeds.{token}"
        return ("direct", [key] if key in seed_plugins else [])

    # 점(.)이 없으면 family로 간주: seeds.<family>.* 검색
    prefix = f"seeds.{token}."
    matches = [k for k in seed_keys if k.startswith(prefix)]
    return ("family", matches)


def fmt_seed_list(seed_keys, limit=40):
    # 보기 좋게 `module.Class` 형태로 표시하고,
    # 너무 길면 앞부분만 보여준 뒤 나머지 개수를 표시합니다.
    names = [k[len("seeds."):] if k.startswith("seeds.") else k for k in seed_keys]
    shown = names[:limit]
    extra = len(names) - len(shown)
    body = "<br>".join(f"`{n}`" for n in shown)
    if extra > 0:
        body += f"<br>… (+{extra} more)"
    return body

# ------------------------------------------------------------
# 3) preset 비교표 + preset별 상세 블록 생성
# ------------------------------------------------------------
# 파일별 한 줄 요약(표에 같이 표시)
one_line_summary = {
    "fast.yaml": "빠른 스모크 테스트용",
    "broad.yaml": "전체 seed를 1회씩 넓게 점검",
    "tox_and_attackers.yaml": "독성/유해성 중심 + attacker 적용",
    "default.yaml": "기본 균형형 기본값",
    "bag.yaml": "다양한 seed/judge 조합의 종합 점검",
    "full.yaml": "폭넓은 심화 점검",
    "notox.yaml": "독성 계열 제외 점검",
    "long_attack_gen.yaml": "긴 반복 생성(고비용) 테스트",
}

md = []
md.append(f"repo_root: `{repo_root}`")
md.append("")
md.append("| preset | 기능 설명 | generations | cap | seed_spec tokens | resolved seeds | unresolved |")
md.append("|---|---|---:|---:|---:|---:|---:|")

details_blocks = []

for preset in presets:
    # 각 yaml 파일을 읽어 run/plugins 섹션을 꺼냅니다.
    data = yaml.safe_load(preset.read_text(encoding="utf-8")) or {}
    run = data.get("run", {}) or {}
    plugins = data.get("plugins", {}) or {}

    # seed_spec을 token 단위로 파싱
    tokens = parse_seed_spec(plugins.get("seed_spec"))
    resolved = []
    unresolved = []

    # token -> 실제 seed key 확장
    for t in tokens:
        _, keys = resolve_seed_token(t)
        if seed_plugins:
            if not keys:
                unresolved.append(t)
            else:
                resolved += keys

    # 중복 제거(순서 유지)
    seen = set()
    resolved = [k for k in resolved if not (k in seen or seen.add(k))]

    gens = run.get("generations", "")
    cap = run.get("soft_seed_prompt_cap", "")
    summary = one_line_summary.get(preset.name, "-")

    # 상단 요약 표 1행 추가
    md.append(
        f"| `{preset.name}` | {summary} | {gens} | {cap} | {len(tokens)} | {len(resolved) if seed_plugins else 'N/A'} | {len(unresolved) if seed_plugins else 'N/A'} |"
    )

    # preset별 상세 설명 블록 생성
    raw = ", ".join(tokens)
    details = []
    details.append(f"### `{preset.name}`")
    details.append(f"- 기능 설명: {summary}")
    details.append(f"- run.generations: `{gens}`")
    details.append(f"- run.soft_seed_prompt_cap: `{cap}`")
    details.append(f"- plugins.seed_spec: `{raw}`" if raw else "- plugins.seed_spec: (empty)")

    if not seed_plugins:
        details.append(f"- plugin_cache.json이 없어 seed_spec 확장을 생략했습니다: `{cache_file}`")
    else:
        details.append(f"- resolved seeds: `{len(resolved)}`")
        if unresolved:
            details.append(f"- unresolved tokens: `{', '.join(unresolved)}`")
        details.append("")
        details.append("<details><summary>seed 목록 펼치기</summary>")
        details.append("")
        details.append(fmt_seed_list(resolved, limit=60) or "(none)")
        details.append("")
        details.append("</details>")

    details_blocks.append("\n".join(details))

# Markdown 렌더링: 요약 표 + 상세 블록
display(Markdown("\n".join(md) + "\n\n" + "\n\n---\n\n".join(details_blocks)))


repo_root: `/Users/selectstar/garak_ko`

| preset | 기능 설명 | generations | cap | seed_spec tokens | resolved seeds | unresolved |
|---|---|---:|---:|---:|---:|---:|
| `bag.yaml` | 다양한 seed/judge 조합의 종합 점검 | 3 | 3 | 39 | 125 | 0 |
| `broad.yaml` | 전체 seed를 1회씩 넓게 점검 | 1 | 3 | 1 | 0 | 1 |
| `default.yaml` | 기본 균형형 기본값 | 3 | 3 | 30 | 119 | 0 |
| `fast.yaml` | 빠른 스모크 테스트용 | 5 | 3 | 18 | 82 | 0 |
| `full.yaml` | 폭넓은 심화 점검 |  | 3 | 24 | 114 | 0 |
| `long_attack_gen.yaml` | 긴 반복 생성(고비용) 테스트 | 100 | 3 | 1 | 1 | 0 |
| `notox.yaml` | 독성 계열 제외 점검 |  | 3 | 12 | 96 | 0 |
| `tox_and_attackers.yaml` | 독성/유해성 중심 + attacker 적용 | 5 | 3 | 9 | 35 | 0 |

### `bag.yaml`
- 기능 설명: 다양한 seed/judge 조합의 종합 점검
- run.generations: `3`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `ansiescape, atkgen.Tox, av_spam_scanning, continuation, dan.Ablation_Dan_11_0, dan.AutoDANCached, dan.DanInTheWild, divergence, encoding.InjectAscii85, encoding.InjectBase16, encoding.InjectBase2048, encoding.InjectBase32, encoding.InjectBase64, encoding.InjectBraille, encoding.InjectEcoji, encoding.InjectHex, encoding.InjectMorse, encoding.InjectNato, encoding.InjectROT13, encoding.InjectUU, encoding.InjectZalgo, exploitation.JinjaTemplatePythonInjection, exploitation.SQLInjectionEcho, exploitation.SQLInjectionSystem, goodside, grandma, latentinjection, leakreplay, lmrc, malwaregen, misleading, packagehallucination, phrasing, realtoxicityprompts, snowball, suffix, tap, topic.WordnetControversial, web_injection`
- resolved seeds: `125`

<details><summary>seed 목록 펼치기</summary>

`ansiescape.AnsiEscaped`<br>`ansiescape.AnsiRaw`<br>`ansiescape.AnsiRawTokenizerHF`<br>`atkgen.Tox`<br>`av_spam_scanning.EICAR`<br>`av_spam_scanning.GTUBE`<br>`av_spam_scanning.GTphish`<br>`continuation.ContinueSlursReclaimedSlurs`<br>`continuation.ContinueSlursReclaimedSlursFull`<br>`dan.Ablation_Dan_11_0`<br>`dan.AutoDANCached`<br>`dan.DanInTheWild`<br>`divergence.Repeat`<br>`divergence.RepeatExtended`<br>`divergence.RepeatedToken`<br>`encoding.InjectAscii85`<br>`encoding.InjectBase16`<br>`encoding.InjectBase2048`<br>`encoding.InjectBase32`<br>`encoding.InjectBase64`<br>`encoding.InjectBraille`<br>`encoding.InjectEcoji`<br>`encoding.InjectHex`<br>`encoding.InjectMorse`<br>`encoding.InjectNato`<br>`encoding.InjectROT13`<br>`encoding.InjectUU`<br>`encoding.InjectZalgo`<br>`exploitation.JinjaTemplatePythonInjection`<br>`exploitation.SQLInjectionEcho`<br>`exploitation.SQLInjectionSystem`<br>`goodside.Davidjl`<br>`goodside.Tag`<br>`goodside.ThreatenJSON`<br>`goodside.WhoIsRiley`<br>`grandma.Slurs`<br>`grandma.Substances`<br>`grandma.Win10`<br>`grandma.Win11`<br>`latentinjection.LatentInjectionFactSnippetEiffel`<br>`latentinjection.LatentInjectionFactSnippetEiffelFull`<br>`latentinjection.LatentInjectionFactSnippetLegal`<br>`latentinjection.LatentInjectionFactSnippetLegalFull`<br>`latentinjection.LatentInjectionReport`<br>`latentinjection.LatentInjectionReportFull`<br>`latentinjection.LatentInjectionResume`<br>`latentinjection.LatentInjectionResumeFull`<br>`latentinjection.LatentInjectionTranslationEnFr`<br>`latentinjection.LatentInjectionTranslationEnFrFull`<br>`latentinjection.LatentInjectionTranslationEnZh`<br>`latentinjection.LatentInjectionTranslationEnZhFull`<br>`latentinjection.LatentJailbreak`<br>`latentinjection.LatentJailbreakFull`<br>`latentinjection.LatentWhois`<br>`latentinjection.LatentWhoisSnippet`<br>`latentinjection.LatentWhoisSnippetFull`<br>`leakreplay.GuardianCloze`<br>`leakreplay.GuardianClozeFull`<br>`leakreplay.GuardianComplete`<br>`leakreplay.GuardianCompleteFull`<br>… (+65 more)

</details>

---

### `broad.yaml`
- 기능 설명: 전체 seed를 1회씩 넓게 점검
- run.generations: `1`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `all`
- resolved seeds: `0`
- unresolved tokens: `all`

<details><summary>seed 목록 펼치기</summary>

(none)

</details>

---

### `default.yaml`
- 기능 설명: 기본 균형형 기본값
- run.generations: `3`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `ansiescape, atkgen, continuation, dan.Ablation_Dan_11_0, dan.AutoDANCached, dan.DanInTheWild, divergence, encoding, exploitation, goodside, grandma, latentinjection, leakreplay, lmrc.Bullying, lmrc.Deadnaming, lmrc.QuackMedicine, lmrc.SexualContent, lmrc.Sexualisation, lmrc.SlurUsage, malwaregen, misleading, packagehallucination, phrasing, promptinject, realtoxicityprompts.RTPBlank, snowball.GraphConnectivity, suffix.GCGCached, tap.TAPCached, topic, web_injection`
- resolved seeds: `119`

<details><summary>seed 목록 펼치기</summary>

`ansiescape.AnsiEscaped`<br>`ansiescape.AnsiRaw`<br>`ansiescape.AnsiRawTokenizerHF`<br>`atkgen.Tox`<br>`continuation.ContinueSlursReclaimedSlurs`<br>`continuation.ContinueSlursReclaimedSlursFull`<br>`dan.Ablation_Dan_11_0`<br>`dan.AutoDANCached`<br>`dan.DanInTheWild`<br>`divergence.Repeat`<br>`divergence.RepeatExtended`<br>`divergence.RepeatedToken`<br>`encoding.InjectAscii85`<br>`encoding.InjectAtbash`<br>`encoding.InjectBase16`<br>`encoding.InjectBase2048`<br>`encoding.InjectBase32`<br>`encoding.InjectBase64`<br>`encoding.InjectBraille`<br>`encoding.InjectEcoji`<br>`encoding.InjectHex`<br>`encoding.InjectLeet`<br>`encoding.InjectMime`<br>`encoding.InjectMorse`<br>`encoding.InjectNato`<br>`encoding.InjectQP`<br>`encoding.InjectROT13`<br>`encoding.InjectSneakyBits`<br>`encoding.InjectUU`<br>`encoding.InjectUnicodeTagChars`<br>`encoding.InjectUnicodeVariantSelectors`<br>`encoding.InjectZalgo`<br>`exploitation.JinjaTemplatePythonInjection`<br>`exploitation.SQLInjectionEcho`<br>`exploitation.SQLInjectionSystem`<br>`goodside.Davidjl`<br>`goodside.Tag`<br>`goodside.ThreatenJSON`<br>`goodside.WhoIsRiley`<br>`grandma.Slurs`<br>`grandma.Substances`<br>`grandma.Win10`<br>`grandma.Win11`<br>`latentinjection.LatentInjectionFactSnippetEiffel`<br>`latentinjection.LatentInjectionFactSnippetEiffelFull`<br>`latentinjection.LatentInjectionFactSnippetLegal`<br>`latentinjection.LatentInjectionFactSnippetLegalFull`<br>`latentinjection.LatentInjectionReport`<br>`latentinjection.LatentInjectionReportFull`<br>`latentinjection.LatentInjectionResume`<br>`latentinjection.LatentInjectionResumeFull`<br>`latentinjection.LatentInjectionTranslationEnFr`<br>`latentinjection.LatentInjectionTranslationEnFrFull`<br>`latentinjection.LatentInjectionTranslationEnZh`<br>`latentinjection.LatentInjectionTranslationEnZhFull`<br>`latentinjection.LatentJailbreak`<br>`latentinjection.LatentJailbreakFull`<br>`latentinjection.LatentWhois`<br>`latentinjection.LatentWhoisSnippet`<br>`latentinjection.LatentWhoisSnippetFull`<br>… (+59 more)

</details>

---

### `fast.yaml`
- 기능 설명: 빠른 스모크 테스트용
- run.generations: `5`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `ansiescape.AnsiRaw, continuation, dan, encoding.InjectBase64, encoding.InjectHex, goodside, av_spam_scanning, leakreplay, lmrc, malwaregen.SubFunctions, malwaregen.TopLevel, packagehallucination, realtoxicityprompts.RTPIdentity_Attack, realtoxicityprompts.RTPProfanity, realtoxicityprompts.RTPSexually_Explicit, realtoxicityprompts.RTPThreat, snowball, web_injection`
- resolved seeds: `82`

<details><summary>seed 목록 펼치기</summary>

`ansiescape.AnsiRaw`<br>`continuation.ContinueSlursReclaimedSlurs`<br>`continuation.ContinueSlursReclaimedSlursFull`<br>`dan.Ablation_Dan_11_0`<br>`dan.AntiDAN`<br>`dan.AutoDAN`<br>`dan.AutoDANCached`<br>`dan.ChatGPT_Developer_Mode_RANTI`<br>`dan.ChatGPT_Developer_Mode_v2`<br>`dan.ChatGPT_Image_Markdown`<br>`dan.DAN_Jailbreak`<br>`dan.DUDE`<br>`dan.DanInTheWild`<br>`dan.DanInTheWildFull`<br>`dan.Dan_10_0`<br>`dan.Dan_11_0`<br>`dan.Dan_6_0`<br>`dan.Dan_6_2`<br>`dan.Dan_7_0`<br>`dan.Dan_8_0`<br>`dan.Dan_9_0`<br>`dan.STAN`<br>`encoding.InjectBase64`<br>`encoding.InjectHex`<br>`goodside.Davidjl`<br>`goodside.Tag`<br>`goodside.ThreatenJSON`<br>`goodside.WhoIsRiley`<br>`av_spam_scanning.EICAR`<br>`av_spam_scanning.GTUBE`<br>`av_spam_scanning.GTphish`<br>`leakreplay.GuardianCloze`<br>`leakreplay.GuardianClozeFull`<br>`leakreplay.GuardianComplete`<br>`leakreplay.GuardianCompleteFull`<br>`leakreplay.LiteratureCloze`<br>`leakreplay.LiteratureClozeFull`<br>`leakreplay.LiteratureComplete`<br>`leakreplay.LiteratureCompleteFull`<br>`leakreplay.NYTCloze`<br>`leakreplay.NYTClozeFull`<br>`leakreplay.NYTComplete`<br>`leakreplay.NYTCompleteFull`<br>`leakreplay.PotterCloze`<br>`leakreplay.PotterClozeFull`<br>`leakreplay.PotterComplete`<br>`leakreplay.PotterCompleteFull`<br>`lmrc.Anthropomorphisation`<br>`lmrc.Bullying`<br>`lmrc.Deadnaming`<br>`lmrc.Profanity`<br>`lmrc.QuackMedicine`<br>`lmrc.SexualContent`<br>`lmrc.Sexualisation`<br>`lmrc.SlurUsage`<br>`malwaregen.SubFunctions`<br>`malwaregen.TopLevel`<br>`packagehallucination.Dart`<br>`packagehallucination.JavaScript`<br>`packagehallucination.Perl`<br>… (+22 more)

</details>

---

### `full.yaml`
- 기능 설명: 폭넓은 심화 점검
- run.generations: ``
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `ansiescape, atkgen, continuation, dan, dan.Ablation_Dan_11_0, encoding, glitch.GlitchFull, goodside, av_spam_scanning, latentinjection, leakreplay.LiteratureClozeFull, leakreplay.LiteratureCompleteFull, lmrc, malwaregen, misleading, packagehallucination, promptinject.HijackHateHumansFull, promptinject.HijackKillHumansFull, promptinject.HijackLongPromptFull, realtoxicityprompts, snowball.GraphConnectivityFull, snowball.PrimesFull, snowball.SenatorsFull, web_injection`
- resolved seeds: `114`

<details><summary>seed 목록 펼치기</summary>

`ansiescape.AnsiEscaped`<br>`ansiescape.AnsiRaw`<br>`ansiescape.AnsiRawTokenizerHF`<br>`atkgen.Tox`<br>`continuation.ContinueSlursReclaimedSlurs`<br>`continuation.ContinueSlursReclaimedSlursFull`<br>`dan.Ablation_Dan_11_0`<br>`dan.AntiDAN`<br>`dan.AutoDAN`<br>`dan.AutoDANCached`<br>`dan.ChatGPT_Developer_Mode_RANTI`<br>`dan.ChatGPT_Developer_Mode_v2`<br>`dan.ChatGPT_Image_Markdown`<br>`dan.DAN_Jailbreak`<br>`dan.DUDE`<br>`dan.DanInTheWild`<br>`dan.DanInTheWildFull`<br>`dan.Dan_10_0`<br>`dan.Dan_11_0`<br>`dan.Dan_6_0`<br>`dan.Dan_6_2`<br>`dan.Dan_7_0`<br>`dan.Dan_8_0`<br>`dan.Dan_9_0`<br>`dan.STAN`<br>`encoding.InjectAscii85`<br>`encoding.InjectAtbash`<br>`encoding.InjectBase16`<br>`encoding.InjectBase2048`<br>`encoding.InjectBase32`<br>`encoding.InjectBase64`<br>`encoding.InjectBraille`<br>`encoding.InjectEcoji`<br>`encoding.InjectHex`<br>`encoding.InjectLeet`<br>`encoding.InjectMime`<br>`encoding.InjectMorse`<br>`encoding.InjectNato`<br>`encoding.InjectQP`<br>`encoding.InjectROT13`<br>`encoding.InjectSneakyBits`<br>`encoding.InjectUU`<br>`encoding.InjectUnicodeTagChars`<br>`encoding.InjectUnicodeVariantSelectors`<br>`encoding.InjectZalgo`<br>`glitch.GlitchFull`<br>`goodside.Davidjl`<br>`goodside.Tag`<br>`goodside.ThreatenJSON`<br>`goodside.WhoIsRiley`<br>`av_spam_scanning.EICAR`<br>`av_spam_scanning.GTUBE`<br>`av_spam_scanning.GTphish`<br>`latentinjection.LatentInjectionFactSnippetEiffel`<br>`latentinjection.LatentInjectionFactSnippetEiffelFull`<br>`latentinjection.LatentInjectionFactSnippetLegal`<br>`latentinjection.LatentInjectionFactSnippetLegalFull`<br>`latentinjection.LatentInjectionReport`<br>`latentinjection.LatentInjectionReportFull`<br>`latentinjection.LatentInjectionResume`<br>… (+54 more)

</details>

---

### `long_attack_gen.yaml`
- 기능 설명: 긴 반복 생성(고비용) 테스트
- run.generations: `100`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `atkgen.Tox`
- resolved seeds: `1`

<details><summary>seed 목록 펼치기</summary>

`atkgen.Tox`

</details>

---

### `notox.yaml`
- 기능 설명: 독성 계열 제외 점검
- run.generations: ``
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `dan, encoding, glitch, goodside, av_spam_scanning, leakreplay, malwaregen, misleading, packagehallucination, promptinject, snowball, web_injection`
- resolved seeds: `96`

<details><summary>seed 목록 펼치기</summary>

`dan.Ablation_Dan_11_0`<br>`dan.AntiDAN`<br>`dan.AutoDAN`<br>`dan.AutoDANCached`<br>`dan.ChatGPT_Developer_Mode_RANTI`<br>`dan.ChatGPT_Developer_Mode_v2`<br>`dan.ChatGPT_Image_Markdown`<br>`dan.DAN_Jailbreak`<br>`dan.DUDE`<br>`dan.DanInTheWild`<br>`dan.DanInTheWildFull`<br>`dan.Dan_10_0`<br>`dan.Dan_11_0`<br>`dan.Dan_6_0`<br>`dan.Dan_6_2`<br>`dan.Dan_7_0`<br>`dan.Dan_8_0`<br>`dan.Dan_9_0`<br>`dan.STAN`<br>`encoding.InjectAscii85`<br>`encoding.InjectAtbash`<br>`encoding.InjectBase16`<br>`encoding.InjectBase2048`<br>`encoding.InjectBase32`<br>`encoding.InjectBase64`<br>`encoding.InjectBraille`<br>`encoding.InjectEcoji`<br>`encoding.InjectHex`<br>`encoding.InjectLeet`<br>`encoding.InjectMime`<br>`encoding.InjectMorse`<br>`encoding.InjectNato`<br>`encoding.InjectQP`<br>`encoding.InjectROT13`<br>`encoding.InjectSneakyBits`<br>`encoding.InjectUU`<br>`encoding.InjectUnicodeTagChars`<br>`encoding.InjectUnicodeVariantSelectors`<br>`encoding.InjectZalgo`<br>`glitch.Glitch`<br>`glitch.GlitchFull`<br>`goodside.Davidjl`<br>`goodside.Tag`<br>`goodside.ThreatenJSON`<br>`goodside.WhoIsRiley`<br>`av_spam_scanning.EICAR`<br>`av_spam_scanning.GTUBE`<br>`av_spam_scanning.GTphish`<br>`leakreplay.GuardianCloze`<br>`leakreplay.GuardianClozeFull`<br>`leakreplay.GuardianComplete`<br>`leakreplay.GuardianCompleteFull`<br>`leakreplay.LiteratureCloze`<br>`leakreplay.LiteratureClozeFull`<br>`leakreplay.LiteratureComplete`<br>`leakreplay.LiteratureCompleteFull`<br>`leakreplay.NYTCloze`<br>`leakreplay.NYTClozeFull`<br>`leakreplay.NYTComplete`<br>`leakreplay.NYTCompleteFull`<br>… (+36 more)

</details>

---

### `tox_and_attackers.yaml`
- 기능 설명: 독성/유해성 중심 + attacker 적용
- run.generations: `5`
- run.soft_seed_prompt_cap: `3`
- plugins.seed_spec: `atkgen.Tox, continuation.ContinueSlursReclaimedSlurs, encoding, lmrc.Anthropomorphisation, lmrc.Bullying, lmrc.Deadnaming, lmrc.Profanity, lmrc.SlurUsage, realtoxicityprompts`
- resolved seeds: `35`

<details><summary>seed 목록 펼치기</summary>

`atkgen.Tox`<br>`continuation.ContinueSlursReclaimedSlurs`<br>`encoding.InjectAscii85`<br>`encoding.InjectAtbash`<br>`encoding.InjectBase16`<br>`encoding.InjectBase2048`<br>`encoding.InjectBase32`<br>`encoding.InjectBase64`<br>`encoding.InjectBraille`<br>`encoding.InjectEcoji`<br>`encoding.InjectHex`<br>`encoding.InjectLeet`<br>`encoding.InjectMime`<br>`encoding.InjectMorse`<br>`encoding.InjectNato`<br>`encoding.InjectQP`<br>`encoding.InjectROT13`<br>`encoding.InjectSneakyBits`<br>`encoding.InjectUU`<br>`encoding.InjectUnicodeTagChars`<br>`encoding.InjectUnicodeVariantSelectors`<br>`encoding.InjectZalgo`<br>`lmrc.Anthropomorphisation`<br>`lmrc.Bullying`<br>`lmrc.Deadnaming`<br>`lmrc.Profanity`<br>`lmrc.SlurUsage`<br>`realtoxicityprompts.RTPBlank`<br>`realtoxicityprompts.RTPFlirtation`<br>`realtoxicityprompts.RTPIdentity_Attack`<br>`realtoxicityprompts.RTPInsult`<br>`realtoxicityprompts.RTPProfanity`<br>`realtoxicityprompts.RTPSevere_Toxicity`<br>`realtoxicityprompts.RTPSexually_Explicit`<br>`realtoxicityprompts.RTPThreat`

</details>

## 2) 선택한 config 상세 보기

아래 셀에서 `CONFIG_NAME`만 바꾸면, 해당 파일의 상세 내용을 확인할 수 있습니다.

### 초보자용 해석 포인트
- `system.lite`: `true`면 가볍고 빠른 실행에 유리
- `run.generations`: 클수록 비용/시간 증가, 탐지 범위도 증가 가능
- `plugins.seed_spec`: 어떤 테스트 seed를 돌릴지 지정
- `plugins.attacker_spec`: 프롬프트 변형(attacker) 적용 방식

처음에는 `fast.yaml` 또는 `broad.yaml`로 시작하고, 목적이 toxicity 점검이면 `tox_and_attackers.yaml`를 권장합니다.


In [5]:
from pathlib import Path
import os

# ------------------------------------------------------------
# 단일 config 상세 확인 셀
# ------------------------------------------------------------
# 목적:
# - 표에서 눈에 띈 config 하나를 골라
#   실제 설정 값(system/run/plugins)을 바로 확인하기 위함

# 1) 작업 경로 보정
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "main.py").exists() else cwd.parent
os.chdir(repo_root)

# 2) 확인할 파일 선택
# 아래 파일명만 바꿔가며 비교하세요.
CONFIG_NAME = "fast.yaml"   # 예: broad.yaml, tox_and_attackers.yaml

# 3) 파일 로드
path = Path("src/garak/configs") / CONFIG_NAME
assert path.exists(), f"파일이 없습니다: {path}"

data = yaml.safe_load(path.read_text(encoding="utf-8")) or {}

# 4) 핵심 섹션 분리
system = data.get("system", {}) or {}
run = data.get("run", {}) or {}
plugins = data.get("plugins", {}) or {}

# 5) 사람이 읽기 쉬운 형태로 출력
print(f"[config] {CONFIG_NAME}")
print("- system:", system)
print("- run:", run)
print("- plugins keys:", sorted(plugins.keys()))
print("- attacker_spec:", plugins.get("attacker_spec", "(none)"))
print("- seed_spec:", plugins.get("seed_spec", "(empty)"))


[config] fast.yaml
- system: {'parallel_attempts': 20, 'lite': True}
- run: {'generations': 5, 'soft_seed_prompt_cap': 3}
- plugins keys: ['extended_judges', 'seed_spec']
- attacker_spec: (none)
- seed_spec: ansiescape.AnsiRaw,continuation,dan,encoding.InjectBase64,encoding.InjectHex,goodside,av_spam_scanning,leakreplay,lmrc,malwaregen.SubFunctions,malwaregen.TopLevel,packagehallucination,realtoxicityprompts.RTPIdentity_Attack,realtoxicityprompts.RTPProfanity,realtoxicityprompts.RTPSexually_Explicit,realtoxicityprompts.RTPThreat,snowball,web_injection
